In [ ]:
# Install dependencies
%pip install anthropic python-dotenv

In [3]:
# Load .env variables
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
# Create an api client
from anthropic import Anthropic

client = Anthropic()

model = "claude-sonnet-4-0"

In [42]:
# Helper functions

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

# Make request
def chat(messages, temperature=1.0, system=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    message = client.messages.create(
        **params
    )
    return message.content[0].text


In [ ]:
# Make a starting list of messages
messages = []

# Add an initial user question of "Define quantum computing in one sentence"
add_user_message(messages, "Define quantum computing in one sentence")

# Pass a list of messages into chat
answer = chat(messages)
# answer

# Take the answer and add it as an assistant message into our list
add_assistant_message(messages, answer)
# messages

# Add in the user's follow up question
add_user_message(messages, "Write another sentence.")
# messages
# Call chat again with the list of messages to get a final aswer
answer = chat(messages)
answer

'Unlike classical computers that use bits representing either 0 or 1, quantum computers use quantum bits (qubits) that can exist in multiple states simultaneously, enabling them to perform many calculations in parallel.'

In [ ]:
# chat exercise:

messages = []

while True:
    # Get user input
    user_input = input("> ")
    print("> ", user_input)

    add_user_message(messages, user_input)
    answer = chat(messages)
    print(answer)
    add_assistant_message(messages, answer)

In [22]:
# How to use system prompts

def chat_tutor(messages):
    system_prompt = """
    You are a patient math tutor.
    Do not directly answer a student's question.
    Guide them to a solution step by step.
    """

    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
        system=system_prompt
    )
    return message.content[0].text

In [28]:
system_prompt = """
    You are a patient math tutor.
    Do not directly answer a student's question.
    Guide them to a solution step by step.
"""

messages = []
add_user_message(messages, "How do I solve 5x+3=2 for x?")
answer = chat(messages, system_prompt)
answer




"I'd be happy to help you solve this equation step by step! Let's work through this together.\n\nFirst, let me ask you: what do you think our goal is when we're solving for x? What do we want x to look like when we're done?\n\nOnce you tell me that, we can think about what we need to do to the equation 5x + 3 = 2 to reach that goal."

In [30]:
# system prompt exercise:

system_prompt = "You are a senior python programmer. Any python code related questions answer as concise as possible. Just with code in a code markdown text that can be print and copy."

messages = []

add_user_message(messages, "Write a Python function that check string for duplicate characters.")
answer = chat(messages, system_prompt)
answer

'```python\ndef has_duplicates(s):\n    return len(s) != len(set(s))\n\n# Alternative with character positions\ndef find_duplicates(s):\n    seen = set()\n    duplicates = set()\n    for char in s:\n        if char in seen:\n            duplicates.add(char)\n        else:\n            seen.add(char)\n    return list(duplicates)\n```'

In [38]:
# Temperature

messages = []
add_user_message(messages, "Generate a one sentence movie idea")

answer = chat(messages, temperature=1.0)
answer

"A deaf librarian discovers that the mysterious sign language conversations she's been translating for late-night visitors are actually instructions for heists targeting the city's most secure locations."

In [44]:
# Streams

messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream :
    for text in stream.text_stream:
        # print(text, end="")
        pass

stream.get_final_message()

ParsedMessage(id='msg_017DSYpYKbkUD5DMUe9L7o1R', container=None, content=[ParsedTextBlock(citations=None, text='FakeDataGen is a synthetic customer database containing 50,000 randomly generated user profiles with fabricated names, addresses, purchase histories, and demographic information designed for software testing and development purposes.', type='text', parsed_output=None)], model='claude-sonnet-4-20250514', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=18, output_tokens=43, server_tool_use=None, service_tier='standard'))

In [ ]:
# Structured data

